# 11 — EUA (CO₂) prices

Builds a **daily CO₂ allowance price** from the EEX primary auctions, needed for the *clean* spark spread.

- **Input:** `data/raw/eex_eua/emission-spot-primary-market-auction-report-*.xlsx`
- **Output:** `data/processed/eua_auction_daily_2020_2025.parquet` (1,826 days: price, last auction date, age of the price)
- **Key points:** only EUA auctions (`T3PA`) that succeeded; results are out by 11:03, so day D uses the last auction up to D-1. Forward fill only, never backward fill.

In [1]:
# 11_eua_ingestion.ipynb - cell 1: setup, file discovery and raw read
import warnings
from pathlib import Path

import pandas as pd

RAW_EUA_DIR = Path("../data/raw/eex_eua")
PROCESSED_DIR = Path("../data/processed")

# EEX files lack a default cell style: openpyxl warns on every read, harmlessly.
warnings.filterwarnings("ignore", message="Workbook contains no default style", category=UserWarning)

# Layout confirmed on all six annual reports (2020-2025):
#   rows 0-4 -> titles and merged group headers ("Prices", "Volumes", ...)
#   row 5    -> column names
#   column A -> always empty
EUA_HEADER_ROW = 5

# We keep only the reference block and the clearing price. The other ~50
# columns are auction statistics and revenue split by member state.
EUA_COLUMNS = {
    "Date": "auction_date",
    "Time": "result_time",          # timestamp of the published result (CET)
    "Auction Name": "auction_name",
    "Contract": "contract",         # T3PA = EUA, EAA3 = aviation allowance (EUAA)
    "Status": "status",
    "Auction Price €/tCO2": "price_eur_t",
}

eua_files = sorted(RAW_EUA_DIR.glob("emission-spot-primary-market-auction-report-*-data.xlsx"))
print(f"{len(eua_files)} annual reports found\n")


def read_auction_report(path):
    """Read one annual EEX primary auction report, keeping only the columns
    we need. No filtering on contract or status here: we want to see every
    row before deciding what to drop."""
    xls = pd.ExcelFile(path, engine="openpyxl")
    # Loud failure if EEX ever adds a second sheet: we would not know which one to read.
    if len(xls.sheet_names) != 1:
        raise ValueError(f"{path.name}: expected 1 sheet, found {xls.sheet_names}")

    df = pd.read_excel(xls, header=EUA_HEADER_ROW)

    # Loud failure if a column gets renamed, instead of a silent KeyError later.
    missing = set(EUA_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: expected columns not found: {missing}")

    df = df[list(EUA_COLUMNS)].rename(columns=EUA_COLUMNS)

    # A real auction row always has a parseable date: this drops blank and footer rows.
    is_auction_row = pd.to_datetime(df["auction_date"], errors="coerce").notna()
    df = df[is_auction_row].copy()

    # Provenance: which file each row came from.
    df["source_file"] = path.name
    return df


eua_raw = pd.concat([read_auction_report(p) for p in eua_files], ignore_index=True)
eua_raw["auction_date"] = pd.to_datetime(eua_raw["auction_date"])
eua_raw["result_time"] = pd.to_datetime(eua_raw["result_time"])

# --- Structural checks -------------------------------------------------------
print(eua_raw.dtypes, "\n")

# Rows and date range per calendar year: spots the winter breaks at a glance.
per_year = eua_raw.groupby(eua_raw["auction_date"].dt.year).agg(
    rows=("auction_date", "size"),
    first=("auction_date", "min"),
    last=("auction_date", "max"),
)
print(per_year, "\n")

# Which products and outcomes are in the files, before any filtering.
print(pd.crosstab(eua_raw["contract"], eua_raw["status"], margins=True))

6 annual reports found

auction_date    datetime64[us]
result_time     datetime64[us]
auction_name               str
contract                   str
status                     str
price_eur_t            float64
source_file                str
dtype: object 

              rows      first       last
auction_date                            
2020           218 2020-01-07 2020-12-14
2021           230 2021-01-29 2021-12-20
2022           222 2022-01-10 2022-12-19
2023           223 2023-01-09 2023-12-18
2024           221 2024-01-15 2024-12-16
2025           213 2025-01-07 2025-12-15 

status    cancelled  successful   All
contract                             
EAA3              0          38    38
T3PA              3        1286  1289
All               3        1324  1327


In [2]:
# 11_eua_ingestion.ipynb - cell 2: keep successful EUA auctions, checks, gap analysis

EUA_CONTRACT = "T3PA"   # EUA general allowance; EAA3 (aviation EUAA) is a different product

is_eua = eua_raw["contract"].eq(EUA_CONTRACT)
is_successful = eua_raw["status"].eq("successful")

# Show what we drop and why, before dropping it.
dropped = eua_raw[~(is_eua & is_successful)]
print(f"dropped {len(dropped)} rows:")
print(dropped.groupby(["contract", "status"]).size().to_string(), "\n")
print(
    dropped.loc[dropped["status"].ne("successful"),
                ["auction_date", "auction_name", "status", "price_eur_t"]]
    .to_string(index=False),
    "\n",
)

# Files are in reverse chronological order: sort ascending once, here.
eua = (
    eua_raw[is_eua & is_successful]
    .sort_values("auction_date")
    .reset_index(drop=True)
)

# --- Blocking checks -----------------------------------------------------------
# One EUA auction per day, so the date can be used as a key.
assert eua["auction_date"].is_unique, "more than one EUA auction on the same date"

# Every successful auction has a strictly positive clearing price.
assert eua["price_eur_t"].notna().all() and (eua["price_eur_t"] > 0).all()

# The result is published on the auction day itself...
assert eua["auction_date"].eq(eua["result_time"].dt.normalize()).all()

# ...and before the MGP gate closure (12:00 of D-1). This is the anti-leakage
# premise for using the auction of day D-1 when forecasting day D.
latest_result = eua["result_time"].dt.time.max()
print(f"latest result time: {latest_result}")
assert latest_result < pd.Timestamp("12:00").time(), "a result was published after 12:00"

print(f"{len(eua)} EUA auctions, "
      f"{eua['auction_date'].min().date()} -> {eua['auction_date'].max().date()}\n")

# --- Gap analysis ----------------------------------------------------------------
# Calendar days between one auction and the next (1 = consecutive days).
eua["gap_days"] = eua["auction_date"].diff().dt.days
# How much the price moved across each gap: what a forward fill would miss.
eua["price_jump"] = eua["price_eur_t"].diff()

print("distribution of gaps (days -> count):")
print(eua["gap_days"].value_counts().sort_index().to_string(), "\n")

longest_gaps = (
    eua.assign(previous_auction=eua["auction_date"].shift(),
               previous_price=eua["price_eur_t"].shift())
    .nlargest(6, "gap_days")
)
print(longest_gaps[["previous_auction", "previous_price", "auction_date",
                    "price_eur_t", "gap_days", "price_jump"]].to_string(index=False))

dropped 41 rows:
contract  status    
EAA3      successful    38
T3PA      cancelled      3 

auction_date              auction_name    status  price_eur_t
  2020-03-17 Auction 3. Period CAP2-EU cancelled          NaN
  2022-03-02 Auction 4. Period CAP3 PL cancelled          NaN
  2022-02-01 Auction 4. Period CAP3 EU cancelled          NaN 

latest result time: 11:03:47.345000
1286 EUA auctions, 2020-01-07 -> 2025-12-15

distribution of gaps (days -> count):
gap_days
1.0     870
2.0     115
3.0     268
4.0      12
5.0      12
6.0       2
7.0       1
21.0      2
22.0      1
28.0      1
46.0      1 

previous_auction  previous_price auction_date  price_eur_t  gap_days  price_jump
      2020-12-14           30.92   2021-01-29        33.51      46.0        2.59
      2023-12-18           66.49   2024-01-15        63.50      28.0       -2.99
      2024-12-16           63.64   2025-01-07        71.02      22.0        7.38
      2021-12-20           74.57   2022-01-10        82.71      21.0  

In [3]:
# 11_eua_ingestion.ipynb - cell 3: daily calendar series, forward fill with age, save

# Same calendar span as gas_mgp_daily_2020_2025.parquet: the first delivery day
# of the perimeter (2021-01-01) needs the value of 2020-12-31.
CALENDAR_START = pd.Timestamp("2020-10-01")
CALENDAR_END = pd.Timestamp("2025-09-30")

# One row per calendar day. Naive midnight timestamps, like gas_day.
# Explicit ns resolution: merge_asof refuses keys with different units.
calendar = pd.DataFrame({
    "date": pd.date_range(CALENDAR_START, CALENDAR_END, freq="D").astype("datetime64[ns]")
})

auctions = eua[["auction_date", "price_eur_t"]].assign(
    auction_date=lambda d: d["auction_date"].astype("datetime64[ns]")
)

# For each calendar date, take the LAST auction on or before that date.
# direction="backward" is what makes this leakage-free: a date can never
# see an auction that happens after it.
eua_daily = pd.merge_asof(
    calendar,
    auctions,
    left_on="date",
    right_on="auction_date",
    direction="backward",
)
eua_daily = eua_daily.rename(columns={
    "auction_date": "eua_last_auction",
    "price_eur_t": "eua_price_eur_t",
})

# Staleness of the value: 0 on auction days, grows through weekends and breaks.
eua_daily["eua_age_days"] = (eua_daily["date"] - eua_daily["eua_last_auction"]).dt.days
eua_daily = eua_daily[["date", "eua_price_eur_t", "eua_last_auction", "eua_age_days"]]

# --- Blocking checks -----------------------------------------------------------
assert len(eua_daily) == 1826, len(eua_daily)
assert eua_daily["date"].is_unique and eua_daily["date"].is_monotonic_increasing
# No NaN: the first calendar day already has an auction behind it (Jan 2020 onwards).
assert eua_daily.notna().all().all(), "missing values after forward fill"
assert (eua_daily["eua_age_days"] >= 0).all()

# On auction days the series must return exactly the auction clearing price.
on_auction_day = eua_daily["eua_age_days"].eq(0)
same_day = eua_daily[on_auction_day].merge(auctions, left_on="date", right_on="auction_date")
assert same_day["eua_price_eur_t"].eq(same_day["price_eur_t"]).all()

print(f"{on_auction_day.sum()} auction days out of {len(eua_daily)} calendar days\n")
print("eua_age_days:")
print(eua_daily["eua_age_days"].describe().round(2).to_string(), "\n")

# Days whose value is more than a week old, per year: where the forward fill costs most.
STALE_THRESHOLD_DAYS = 7
is_stale = eua_daily["eua_age_days"] > STALE_THRESHOLD_DAYS
print(f"days with eua_age_days > {STALE_THRESHOLD_DAYS}, per year:")
print(eua_daily[is_stale].groupby(eua_daily["date"].dt.year).size().to_string(), "\n")

print("stalest day:")
print(eua_daily.loc[eua_daily["eua_age_days"].idxmax()].to_string(), "\n")

# --- Persist and verify the round-trip -------------------------------------------
out_path = PROCESSED_DIR / "eua_auction_daily_2020_2025.parquet"
eua_daily.to_parquet(out_path, index=False)

check = pd.read_parquet(out_path)
# check_dtype=False: pandas 3 may read datetimes back at a different resolution (us vs ns).
pd.testing.assert_frame_equal(check, eua_daily, check_dtype=False)

print(f"saved {check.shape[0]} rows x {check.shape[1]} cols -> {out_path.name}")

1076 auction days out of 1826 calendar days

eua_age_days:
count    1826.00
mean        1.65
std         4.92
min         0.00
25%         0.00
50%         0.00
75%         1.00
max        45.00 

days with eua_age_days > 7, per year:
date
2020    10
2021    32
2022    14
2023    14
2024    22
2025     6 

stalest day:
date                2021-01-28 00:00:00
eua_price_eur_t                   30.92
eua_last_auction    2020-12-14 00:00:00
eua_age_days                         45 

saved 1826 rows x 4 cols -> eua_auction_daily_2020_2025.parquet
